# Scrape ATP Player Basics

In [1]:
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
import numpy as np
from geopy.geocoders import Nominatim

## 1. Scape/update ATP Profile Links

In [2]:
# load txt files
def load_txt(file):
    with open(file, 'r', encoding='utf-8-sig') as f:
        attributes = f.readline()[:-1].split(',')
        df = pd.DataFrame([line[:-1].split(',') for line in f], columns = attributes)
    f.close()
    return df

In [ ]:
# update links according to tennisabstract

chromedrive_path = "D:\chromedriver.exe"
s = Service(chromedrive_path)
driver = webdriver.Chrome(service=s)
driver.set_window_rect(x=10, y=10, width=100, height=200)

prefix = 'https://www.tennisabstract.com/cgi-bin/player.cgi?p='
atplinks = []

# open mp txt, get player name and search for atplink
with open('mp.txt', 'r', encoding='utf-8-sig') as f:
    for i, line in enumerate(f):
        # start from, MODIFY!!!
        if i >= 4900:
            name = line[4:].replace(' ', '')
            driver.get(prefix+name)
            driver.refresh()
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            try:
                atplinks.append(soup.find('a', {'title': 'Profile at atpworldtour.com'}).get('href').replace('world', ''))
            except:
                pass
            print('Finished:', i + 1)
f.close()

# open txt
with open('m/basics/atpprofilelinks.txt', 'r', encoding='utf-8-sig') as f:
    links = [line[:-1] for line in f]
f.close()

# add new links
for atplink in atplinks:
    if atplink not in links:
        links.append(atplink)

# update links
with open('m/basics/atpprofilelinks.txt', 'w', encoding='utf-8-sig') as f:
    for link in links:
        if link != '':
            f.write(link)
            f.write('\n')
f.close()

In [5]:
# update links according to atp ranking

prefix = 'https://www.atptour.com/en/rankings/singles?RankRange=0-100&WeekDate='
head = 'https://www.atptour.com'

# open txt
df = load_txt('m/basics/atpprofilebasics.txt')
pl = [p for p in df['Player']]

with open('m/basics/atpprofilelinks.txt', 'r', encoding='utf-8-sig') as f:
    links = [line[:-1] for line in f]
f.close()

chromedrive_path = "D:\chromedriver.exe"
s = Service(chromedrive_path)
driver = webdriver.Chrome(service=s)
driver.set_window_rect(x=10, y=10, width=1500, height=800)

# date list, MODIFY DATE HERE!!!
for date in ['2024-05-06']:
    driver.get(prefix + date)
    driver.refresh()
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    trs = soup.find('table', {'class':'mega-table desktop-table non-live'}).find_all('tr')
    for tr in trs[1:]:
        tds = tr.find_all('td')
        if len(tds) == 8:
            lis = tds[1].find_all('li')
            if lis[2].text.strip().replace('-', ' ') not in pl:
                # add new link
                links.append(head+lis[2].a.get('href'))
                pl.append(lis[2].text.strip().replace('-', ' '))
                print('Found:', lis[2].text.strip().replace('-', ' '))
    print('Finish:', date)
f.close()

# update links
with open('m/basics/atpprofilelinks.txt', 'w', encoding='utf-8-sig') as f:
    for link in set(links):
        if link != '':
            f.write(link)
            f.write('\n')
f.close()

Found: Mariano Navone
Found: Fabian Marozsan
Found: Luciano Darderi
Found: Flavio Cobolli
Found: Jakub Mensik
Found: Alex Michelsen
Found: Arthur Cazaux
Found: Luca Nardi
Found: Aleksandar Kovacevic
Found: Francisco Comesana
Finish: 2024-05-06


## Scrape/update ATP Profile Information

In [8]:
# UPDATE new basics based on new links in atprofilelinks

with open('m/basics/atpprofilelinks.txt', 'r', encoding='utf-8-sig') as f:
    links = [line[:-1] for line in f]
f.close()

dict = load_txt('m/basics/atpprofilebasics.txt').to_dict('list')

chromedrive_path = "D:\chromedriver.exe"
s = Service(chromedrive_path)
driver = webdriver.Chrome(service=s)
driver.set_window_rect(x=10, y=10, width=100, height=200)

for i, link in enumerate(links):
    if link not in dict['ATP Link']:
        driver.get(link)
        driver.refresh()
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        # add name
        try:
            dict['Player'].append(soup.find('div', {'class': 'player_name'}).text.strip().replace('-', ' '))
        except:
            print(link, 'not found')
            continue  # run next iteration
        # add link
        dict['ATP Link'].append(link)
        # add birthday
        try:
            txt = soup.find('span', {'class': 'table-birthday'}).text.split('.')
            birthday = txt[0][-4:] + '-' + txt[1] + '-' + txt[2][:2]
        except:
            birthday = ''
        dict['Birthday'].append(birthday)
        # add birthplace, plays
        try:
            birthplace_plays = soup.find_all('div', {'class': 'table-value'})
            dict['Birthplace'].append(birthplace_plays[0].text.strip('\n ').replace(',', ' -'))
            dict['Plays'].append(birthplace_plays[1].text.strip().replace(',', ' -'))
        except:
            dict['Birthplace'].append('')
            dict['Plays'].append('')
        # add turned pro
        try:
            dict['Turned Pro'].append(soup.find_all('div', {'class': 'table-big-value'})[1].text.strip())
        except:
            dict['Turned Pro'].append('')
        # add weight, height
        try:
            dict['Weight (kg)'].append(soup.find('span', {'class': 'table-weight-kg-wrapper'}).text.strip('()kg'))
        except:
            dict['Weight (kg)'].append('')
        try:
            dict['Height (cm)'].append(soup.find('span', {'class': 'table-height-cm-wrapper'}).text.strip('()cm'))
        except:
            dict['Height (cm)'].append('')
        print('Finished:', i+1)
        # add birthplace details, matched region, lat, lon
        place = birthplace_plays[0].text.strip('\n ')
        geolocator = Nominatim(user_agent='tennis-vis')
        try:
            location = geolocator.geocode(place)
            dict['Birthplace_matched'].append(location.address.replace(',', ' -'))
            dict['Birthplace_lat'].append(location.latitude)
            dict['Birthplace_lon'].append(location.longitude)
            print(location.address.replace(',', ' -'), location.latitude, location.longitude)
        except:
            try:
                location = geolocator.geocode(place.split(',')[0]+','+place.split(',')[1])
                dict['Birthplace_matched'].append(location.address.replace(',', ' -'))
                dict['Birthplace_lat'].append(location.latitude)
                dict['Birthplace_lon'].append(location.longitude)
                print(location.address.replace(',', ' -'), location.latitude, location.longitude)
            except:
                try:
                    location = geolocator.geocode(place.split(',')[0])
                    dict['Birthplace_matched'].append(location.address.replace(',', ' -'))
                    dict['Birthplace_lat'].append(location.latitude)
                    dict['Birthplace_lon'].append(location.longitude)
                    print(location.address.replace(',', ' -'), location.latitude, location.longitude)
                except:
                    try:
                        location = geolocator.geocode(place.split(',')[1])
                        dict['Birthplace_matched'].append(location.address.replace(',', ' -'))
                        dict['Birthplace_lat'].append(location.latitude)
                        dict['Birthplace_lon'].append(location.longitude)
                        print(location.address.replace(',', ' -'), location.latitude, location.longitude)
                    except:
                        try:
                            location = geolocator.geocode(place.split(',')[2])
                            dict['Birthplace_matched'].append(location.address.replace(',', ' -'))
                            dict['Birthplace_lat'].append(location.latitude)
                            dict['Birthplace_lon'].append(location.longitude)
                            print(location.address.replace(',', ' -'), location.latitude, location.longitude)
                        except:
                            dict['Birthplace_matched'].append('')
                            dict['Birthplace_lat'].append('')
                            dict['Birthplace_lon'].append('')
                            print('','','')

df = pd.DataFrame(dict)
df.to_csv('m/basics/atpprofilebasics.txt', index = False, sep = ',', encoding='utf-8-sig')

https://www.atptour.com/en/players/mariano-navone/n0bs/overview not found
https://www.atptour.com/en/players/alex-michelsen/m0qi/overview not found
https://www.atptour.com/en/players/aleksandar-kovacevic/k0az/overview not found
https://www.atptour.com/en/players/flavio-cobolli/c0e9/overview not found
https://www.atptour.com/en/players/fabian-marozsan/m0ci/overview not found
https://www.atptour.com/en/players/arthur-cazaux/c0h0/overview not found
https://www.atptour.com/en/players/luca-nardi/n0bg/overview not found
https://www.atptour.com/en/players/jakub-mensik/m0ni/overview not found
https://www.atptour.com/en/players/luciano-darderi/d0fj/overview not found
https://www.atptour.com/en/players/francisco-comesana/c0df/overview not found


In [5]:
df = load_txt('m/matches/txt/Frank Parker.txt')

In [6]:
df.head(10)

,Date,Tournament,Sets,Surface,Rd,Rk,vRk,W,tRk,vtRk,...,v2nd%,BPCnv,TP,Aces,DFs,SP,1SP,2SP,vA,Time
0,7-Mar-1971,Hampton,3,Hard,R32,,,0,,,...,,,,,,,,,,
1,29-Aug-1968,US Open,5,Grass,R64,,,0,,5,...,,,,,,,,,,
2,0-Aug-1949,US Open,5,Grass,SF,,,0,,,...,,,,,,,,,,
3,0-Aug-1949,US Open,5,Grass,QF,,,1,,,...,,,,,,,,,,
4,0-Aug-1949,US Open,5,Grass,R16,,,1,,,...,,,,,,,,,,
5,0-Aug-1949,US Open,5,Grass,R32,,,1,,,...,,,,,,,,,,
6,0-Aug-1949,US Open,5,Grass,R64,,,1,,,...,,,,,,,,,,
7,0-Jun-1949,Wimbledon,5,Grass,QF,,,0,,,...,,,,,,,,,,
8,0-Jun-1949,Wimbledon,5,Grass,R16,,,1,,,...,,,,,,,,,,
9,0-Jun-1949,Wimbledon,5,Grass,R32,,,1,,,...,,,,,,,,,,
